<a href="https://colab.research.google.com/github/covillarreal/Aprendizaje-Automatico-2/blob/main/TP1_AA2_VILLARREAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 AA1 Aprendizaje no supervisado

**CONSTANZA VILLARREAL**

## Indicaciones básicas

0) Debe usar este notebook como template para su entrega. Haga una copia y comience a completar las consignas.

1) Cada uno debe completar las consignas indicadas en este notebook.

2)
3) No pueden repetir el mismo dataset que ya haya definido un compañero.

4) copias explícitas de secciones enteras del trabajo de otro será penalizado disminuyendo su puntuación.

5) No se olvide de añadir las fuentes de inspiración de su código (blogs, prompts de chatgpt o similar).

6) Además de todo el código que agregue, es importante que sepa interpretarlo. Agregue texto explicativo en cada sección. Esto le ayudará al momento del coloquio / parcial

7) Revise las fecha límite de entrega de este trabajo


##**Tarea: Entrenamiento y evaluación de clasificadores**  
**Objetivo**: Aplicar un modelo de clasificación a un dataset de su elección, procesar dicho dataset para poder usarlo para entrenamiento, indicar y compartir todos los recursos utilizados, evaluar su rendimiento.




## **Instrucciones**:

#1. **Selección del Dataset**  🪄
   - Elijan un dataset de UCI ML Repository del siguiente enlace: https://archive.ics.uci.edu/datasets?Task=Clustering&skip=130&take=10&sort=desc&orderBy=Relevance&search=
   - Requisitos:  
     - Debe tener al menos 4 variables numéricas continuas.  
     - Idealmente, que las features tengan distintas escalas o unidades ( no excluyente).
     - Revisar en el foro de la tarea que dicho dataset no haya sido ya elegido por otra persona.
     - Postee en el foro de la tarea el dataset que eligió. Continue al siguiente punto.  

DATASET UTILIZADO: [WINE](https://archive.ics.uci.edu/dataset/109/wine)

* Descripción: Diferencia tipos de vino a partir de características químicas.

* Variables: 13 variables numéricas que describen componentes químicos del vino

* Instancias: 178

* Clases: 3 tipos de vino (multiclase)

* Valores nulos: No tiene

## Resolución:

Importo librerias y configuraciones generales:

In [ ]:
import numpy as np               # arreglos, matemáticas
import pandas as pd              # tablas, datos

import matplotlib.pyplot as plt  # graficos simples
import seaborn as sns            # graficos estadisticos

from sklearn.pipeline import Pipeline                  # encadenar pasos
from sklearn.compose import ColumnTransformer          # transformar columnas
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # escalar, codificar

from sklearn.model_selection import train_test_split, cross_val_score  # dividir datos, validación cruzada
from sklearn.linear_model import LogisticRegression     # clasificación
from sklearn.neighbors import KNeighborsClassifier       # clasificación
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, precision_score, recall_score # métricas, evaluación

In [ ]:
# Nombres de las columnas (según documentación de UCI)
columnas = [
    'Clase','Alcohol','Ácido Málico','Ceniza','Alcalinidad de la Ceniza',
    'Magnesio','Fenoles Totales','Flavonoides','Fenoles No Flavonoides',
    'Proantocianinas','Intensidad de Color','Matiz',
    'OD280/OD315 del Vino','Prolina'
]
# Cargar el dataset
df = pd.read_csv('wine.data.csv', header=None, names=columnas)

# Vista previa
df.head()



Primero importo las librerías necesarias: pandas (como pd), que utilizo para manipular y analizar datos; matplotlib.pyplot (como plt), que me sirve para crear gráficos; y seaborn (como sns), que es una librería que mejora la visualización de los datos, haciéndola más estética y sencilla de usar.

Luego, defino una lista de nombres de columnas, que corresponden a las características de un conjunto de datos relacionado con vinos que he sacado la informacion de la página proporcionada. Estos nombres los asigno a la variable columnas.

Después, cargo el conjunto de datos llamado 'wine.data.csv' que seleccione para trabajar utilizando pandas con el método read_csv(). Este método lee el archivo CSV y lo convierte en un DataFrame (una estructura de datos tabular) al que le asigno los nombres de las columnas definidos previamente. Finalmente, imprimo las primeras filas del DataFrame con df.head() para ver cómo luce el conjunto de datos cargado.


In [ ]:
df.head()


## 2. **Análisis exploratorio (previo al modelado)**   🔎📊
   - Describan las variables (media, distribución, outliers).  
   - Visualizen:  
     - Histogramas o boxplots para ver distribuciones.  
     - Gráficos de dispersión (scatterplots) entre features y target.  
   - Describan si observan o no relaciones entre algunas variables.  


## Resolución:

Con esta línea de código, lo que hago es generar una descripción estadística del DataFrame df usando el método describe(). Este método me proporciona información clave sobre cada una de las columnas numéricas del conjunto de datos, como:

* Cuenta (count): El número de valores no nulos que tiene cada columna.

* Media (mean): El valor promedio de cada columna.

* Desviación estándar (std): Cuánto varían los valores de cada columna respecto a la media.

* Mínimo (min): El valor más bajo de cada columna.

* Cuartiles (25%, 50%, 75%): Los valores que dividen los datos en cuartiles. El 50% es la mediana, y los otros dos valores son el primer y tercer cuartil.

* Máximo (max): El valor más alto de cada columna.

Esto me ayuda a obtener una visión general rápida de cómo están distribuidos los datos y si existen valores atípicos o alguna columna que tenga datos extraños o faltantes.

In [ ]:
df.describe() #descripcion estadistica

📌 Conclusiones clave

Hay varias variables con valores atípicos o asimetrías, como:

* Ácido Málico, Alcalinidad de la Ceniza, Magnesio, Color, y sobre todo Prolina.

Esto sugiere que:

* Hay vinos químicamente muy distintos entre sí.

* Algunas clases pueden tener características químicas mucho más marcadas que otras.

Estos outliers no necesariamente son errores, pero es importante escalarlos y analizarlos con cuidado antes de aplicar modelos.

In [ ]:
df.drop('Clase', axis=1).hist(figsize=(15, 12), bins=15)
plt.suptitle("Distribuciones de las Variables Numéricas")
plt.show()

Con este bloque de código, lo que hago es lo siguiente:

Primero, elimino la columna 'Clase' del DataFrame df utilizando el método drop(). Como no me interesa esta columna para el análisis de distribuciones numéricas, la dejo fuera. La opción axis=1 indica que quiero eliminar una columna, no una fila.

Luego, genero un histograma para cada una de las variables numéricas restantes del DataFrame. Para ello, uso el método hist(), que crea un histograma para cada columna numérica. La opción figsize=(15, 12) especifica el tamaño de la figura para que los gráficos sean lo suficientemente grandes y fáciles de leer. La opción bins=15 indica que cada histograma tendrá 15 intervalos (o "bins").

Finalmente, añado un título general a la figura utilizando plt.suptitle() y luego muestro los gráficos con plt.show(). Esto me permite ver cómo se distribuyen los datos de cada una de las variables numéricas en el conjunto de datos.

Se puede observar rapidamente que se encuentran valores extremos pero no en mucha cantidad. En Ceniza podemos observar que es el que mas valores atipicos tiene.
Ahora analizaremos con el Z-Score que es una forma de medir cuántos desvíos estándar está un valor respecto a la media. Un Z-score muy alto suele considerarse outlier

In [ ]:
from scipy import stats

x = df.drop('Clase', axis=1)
z_scores = stats.zscore(x)

In [ ]:
# detectar outliers (3+ sigmas, 99% de los datos)
outliers = np.abs(z_scores) > 3

# contar filas con 1+ outliers
outliers_per_row = outliers.any(axis=1).sum()

print('Filas con 1+ outliers: ', outliers_per_row)
print('Total de outliers:     ', outliers.sum().sum())
print('Detalle por feature:', outliers.sum(axis=0))

Como vimos en el grafico anteriormente, podemos ver de nuevo que tengo en total 11 outliers, solo 2 con mas de 1 que son:
* Ceniza tiene 3 outliers

* Magnesium tiene 2

El resto solo 1, entonces los valores estan dispersos y puede que sea un ruido normal, no errores.
Tambien puedo decir que 10 filas con al menos un outlier de un total de 178 -> poco más del 5% -> no parece excesivo.

Tomare la decisión de no eliminarlos por el momento

In [ ]:
correlation = x.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', annot_kws={"size": 8})
plt.title('Matriz de Correlación', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8)   # rotar los label a 45 grados y alinear su texto
plt.yticks(rotation=45, va='top', fontsize=8)     # rotar los label a 45 grados y alinear su texto
plt.show()

**DESCRIPCIÓN SENCILLA DE LAS FEATURES PRINCIPALES:**
* Alcohol:
  
  → Representa el porcentaje de alcohol en el vino.
  
  → Afecta al cuerpo, sabor y fuerza del vino.

* Ácido Málico:
  
  → Es un ácido natural presente en las uvas.
  
  → Aporta acidez, pero disminuye a medida que la uva madura.

* Flavonoides:

  → Son compuestos que influyen en el sabor, el color y los beneficios para la salud del vino.

  → Más flavonoides suelen estar relacionados con vinos más intensos o de mayor calidad.

* Prolina:

  → Es un aminoácido que aparece en mayor cantidad en vinos más maduros.

  → Puede estar asociado con la calidad del vino.

* OD280/OD315 del Vino:

  → Es una medida técnica que evalúa la cantidad de ciertos compuestos fenólicos.

  → Estos compuestos están relacionados con el color y sabor del vino.

* Clase:

  → Es la categoría o tipo de vino al que pertenece (por ejemplo, tipo 1, 2 o 3).

*Esta informacion la busqué en internet y ChatGpt para tener mayor conocimiento del tema y analizar con un poco mas de contexto*

**Correlaciones Positivas Fuertes (mayores a 0.7)**

Estas variables aumentan juntas:

* Fenoles Totales y Flavonoides: 0.86

  * Muy lógico, ya que los flavonoides son un tipo de fenol.

* OD280/OD315 del Vino y Flavonoides: 0.79

  * Es probable que esta medición esté captando la presencia de flavonoides.

* Prolina y OD280/OD315 del Vino: 0.64

  * Tal vez ambas características estén relacionadas con la madurez de la uva o calidad del vino.

* Prolina y Fenoles Totales: 0.70

* Magnesio y Fenoles Totales: 0.64

**Correlaciones Negativas Fuertes (menores a -0.5)**

Estas variables aumentan en direcciones opuestas:

* Ácido Málico y Flavonoides: -0.56

  * Puede indicar un balance químico: cuando hay más flavonoides, menos ácido málico.

* Ácido Málico y OD280/OD315 del Vino: -0.56

* Intensidad de Color y Flavonoides: -0.52

  * Puede ser que ciertos tipos de flavonoides no están relacionados con el color.

**Correlaciones Medias (0.4 a 0.6)**
* Alcohol y Prolina: 0.64

* Alcohol y Intensidad de Color: 0.55

  * Parece que hay una ligera relación entre alcohol y color, interesante desde el punto de vista enológico.

* Fenoles No Flavonoides y Fenoles Totales: 0.61

**Variables poco correlacionadas (casi 0)**

Ceniza, Matiz y Ácido Málico están muy cerca de 0 con muchas variables.

Esto no significa que no sean útiles, pero pueden no estar linealmente relacionadas con las otras.

In [ ]:
# Gráfico de pares con nombres de columnas
sns.pairplot(df[['Alcohol','Flavonoides', 'Prolina', 'Ácido Málico', 'OD280/OD315 del Vino', 'Clase']], hue='Clase')
plt.suptitle("Relación entre variables clave y la clase", y=1.02)
plt.show()

Como podemos observar se sigue reflejando la correlacion que analicé anteriormente, las correlaciones positivas, negativas y la casi nula. En general se puede ver que la distribucion entre clases es bastante clara

En este bloque de código, lo que hago es generar una matriz de gráficos de dispersión (scatter plots) entre algunas variables del DataFrame, con la ayuda de seaborn y su función pairplot().

Primero, selecciono un subconjunto de las columnas del DataFrame df que quiero analizar. Esas son las variables que se van a comparar entre sí en los gráficos.

Luego, utilizo la función sns.pairplot() para crear los gráficos de dispersión entre todas las combinaciones posibles de estas columnas seleccionadas. Además, paso el argumento hue='Clase' para que cada punto de los gráficos se coloree de acuerdo con la columna 'Clase', que es la variable objetivo (en este caso, la clase de vino). Esto me permite ver cómo se agrupan las diferentes clases de vino según estas características.

Finalmente, añado un título general con plt.suptitle("Relación entre algunas variables clave y la clase") y muestro la matriz de gráficos con plt.show().

El resultado es una visualización que me ayuda a entender cómo se relacionan las variables seleccionadas entre sí y cómo estas relaciones difieren entre las distintas clases de vino.

## 3. **Preprocesamiento**  
   - Limpieza: Manejen missing values (eliminar, imputar) y outliers (si es necesario).  
   - Limpieza: indique cuáles features descarta. Justifique.
   - Indique si usará o no variables categóricas. Justifique. Realice su preprocesamiento adeucado.
   - Otros pasos que crea conveniente para pre-procesar el dataset (mencione y explique)


Detalla las caracteristicas del dataset como nro de variables, nro de filas o instancias, si el problema es clasificacion binaria o multiclase, y otras características que crea conveniente.

Realice la división de datos (entrenamiento / testeo / CV según corresponda)

## Resolución:

**CARACTERISTICAS GENERALES DEL DATASET**
* Descripción: Diferencia tipos de vino a partir de características químicas.

* Variables: 13 variables numéricas que describen componentes químicos del vino

* Instancias: 178

* Clases: 3 tipos de vino (multiclase)

* Valores nulos: No tiene

In [ ]:
# Revisamos si hay valores nulos en el dataset
df.isnull().sum()

Con esta línea de código, lo que hago es revisar si hay valores nulos (es decir, datos faltantes) en el DataFrame df. Utilizo el método isnull() para identificar las celdas que contienen valores nulos, lo que me devuelve un DataFrame de valores booleanos (True para los valores nulos y False para los valores no nulos).

Luego, con .sum(), sumo los valores True (que representan los valores nulos) por cada columna. Esto me muestra cuántos valores nulos hay en cada columna del conjunto de datos.

De esta manera, puedo identificar rápidamente si existe algún problema con los datos faltantes y saber en qué columnas podría necesitar realizar algún tratamiento o limpieza de datos.

1. No se encuentran valores faltantes.
2. No voy a eliminar outliers por la justificacion que di anteriormente al momento de identificarlos.
3. Con el objetivo de simplificar el modelo (ya que no conozco en profundidad todas las variables químicas), decidí eliminar las siguientes columnas:

  * Fenoles Totales: Muy correlacionada con Flavonoides (r = 0.86), me quedo con Flavonoides para evitar redundancia.

  * OD280/OD315 del Vino: Alta correlación con Flavonoides y Prolina, por lo que ya estaría representada de forma indirecta.

  * Prolina: También muy relacionada con otras variables como Fenoles y OD280/OD315.

  * Ceniza: Baja correlación con casi todas las variables, no aporta mucho valor al modelo.

  * Matiz: Mismo caso que Ceniza, poca relación con las demás columnas.

  * Ácido Málico: Tiene algunas correlaciones negativas pero en general poco clara su utilidad en un modelo simple.

In [ ]:
# Lista de columnas a eliminar
columnas_eliminar = [
    'Fenoles Totales',
    'OD280/OD315 del Vino',
    'Prolina',
    'Ceniza',
    'Matiz',
    'Ácido Málico'
]

# Creo un nuevo DataFrame sin esas columnas
df_filtrado = df.drop(columns=columnas_eliminar)

# Verifico el nuevo DataFrame
print("Columnas restantes:", df_filtrado.columns.tolist())




A continuación, aplico el escalado a las características con StandardScaler de sklearn. Este es un tipo de normalización que transforma los datos de manera que tengan media cero y desviación estándar uno. Esto es útil para asegurar que todas las características tengan la misma escala, evitando que algunas variables dominen a otras debido a su magnitud.

Utilizo scaler.fit_transform(X) para ajustar el escalador a los datos de X (es decir, calcula la media y desviación estándar) y luego transformar esos datos, escalándolos. El resultado es un nuevo conjunto de datos, X_scaled, que contiene las mismas características pero ahora en una escala estándar.


In [ ]:
# Separar features (X) y variable objetivo (y)
X = df_filtrado.drop(columns=['Clase'])
y = df_filtrado['Clase']

# Crear el escalador
scaler = StandardScaler()

# Ajusto solo con el conjunto de entrenamiento
scaler.fit(X)

# Transformo los datos
X_scaled = scaler.transform(X)

# Verifico que la media sea 0 y la desviación 1 (como debe ser tras escalar)
print("Media de X_scaled:", np.mean(X_scaled, axis=0))
print("Desvío estándar de X_scaled:", np.std(X_scaled, axis=0))


Los valores de media están prácticamente en cero y el desvío estándar está justo en 1.

Esto confirma que la normalización se realizó correctamente y que el escalador está funcionando bien.

4. No hay variables categóricas entre las features, solo numéricas. Tengo "clase" que es el target pero tiene números y Logistic Regression lo puede manejar





## 4. **Clasificación con logistic regression**    🎯🧩

En este apartado entrenará un clasificador con la librería sklearn usando logistic regresion.
   

### 4.1 Entrenamiento y evaluación



```
# # Cargar datos: Separar features (X) y variable objetivo (y)-> ya lo realice en el preprocesamiento
X = df_filtrado.drop(columns=['Clase'])
y = df_filtrado['Clase']
```



In [ ]:
print("Clases originales en y:", np.unique(y, return_counts=True))

In [ ]:
# Filtrar clases 1 y 2 antes de dividir
binary_mask = (y == 1) | (y == 2)
X_binary = X[binary_mask]
y_binary = y[binary_mask]
y_binary = y_binary.astype(int)

# Verificar las clases disponibles
print("Clases disponibles en y_binary:", np.unique(y_binary, return_counts=True))

# Dividir los datos en conjunto de entrenamiento y prueba de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

# Crear y entrenar el clasificador
clf = LogisticRegression(max_iter=1000) #aumente el numero de iteracion
clf.fit(X_train, y_train)

# Realizar predicciones
y_pred = clf.predict(X_test)

Decidí hacerlo de esta manera porque la regresión logística trabaja con dos clases, y mi dataset tenía tres. Para que el modelo funcionara sin problemas, filtré las clases 1 y 2, dejando fuera la clase 3 y convirtiendo el problema en una clasificación binaria.
Además, dividí los datos asegurándome de que la proporción de clases se mantuviera equilibrada en entrenamiento y prueba. También aumenté el número de iteraciones (1000), para darle al modelo más oportunidades de mejorar sus predicciones.
Con este enfoque, adapté la regresión logística al dataset y mejoré la estabilidad del modelo.

### 4.2 Métricas de evaluación

Muestre el desempeño en el conjunto de datos de entrenamiento y testeo.
Mencione las métricas utilizadas. No se olvide de mostrar la matriz de confusión.
Explique los resultados obtenidos.

In [ ]:
# Evaluar el modelo
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Interpretación:

- **Matriz de Confusión**:
  - 16 Verdaderos Negativos
  - 17 Verdaderos Positivos
  - 5 Falso Positivo (predijo clase 2, pero era clase 1).
  - 1 Falsos Negativos (predijo clase 1, pero era clase 2).
  
- **Exactitud (Accuracy)**: El modelo acertó en el 84.6% de los casos.
- **Precisión (Precision)**: De todas las predicciones positivas, el 77% fueron correctas en clase 1 y el 94% en clase 2
- **Sensibilidad (Recall)**: El modelo identificó correctamente el 94% de los casos positivos en clase 1 y 76% en clase 2.
- **F1-Score**: Una medida de balance entre precisión y sensibilidad, alcanzando 0.85 en clase 1 y 0.84 en clase 2

El modelo tuvo un buen rendimiento general, con un 84.6% de precisión en la clasificación. Sin embargo, cometió algunos errores, especialmente confundiendo algunos ejemplos de la clase 2 con la clase 1. La sensibilidad en clase 1 (94%) fue alta, lo que indica que identificó correctamente la mayoría de los casos positivos, mientras que en clase 2 fue menor (76%).
En resumen, el modelo funciona bien, pero podría mejorar la identificación de la clase 2 para reducir los falsos negativos.


## **5. Otro clasificador**

### 5.1 Elija otro modelo para entrenar un clasificador

Elija alguno de los modelos vistos: kNN, SVM o MLP para entrenar un clasificador usando los parámetros por defecto de sklearn.

Justifique su elección.

Elegí **K-Nearest Neighbors (KNN)** porque es un modelo fácil de entender que clasifica los datos según sus vecinos más cercanos. Como mi conjunto de datos tiene tres clases bien definidas, KNN funciona bien, ya que agrupa instancias similares. Aunque puede ser afectado por el ruido, elegir bien el valor de k y normalizar los datos ayuda a mejorar su rendimiento.


### 5.2 Entrenamiento del modelo.

Primero debo definir x e y que ya lo hice anteriormente:



```
# X = df_filtrado.drop(columns=['Clase'])
y = df_filtrado['Clase']
```





Para determinar el número ideal de clústers se debería utilizar Silhoutte Score pero en mi caso voy a utilizar K=3 ya que es el numero de clases que tengo en el dataset. Luego hare el analisis final cambiando los valores de K.



In [ ]:
# Divido el dataset en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Creo el modelo KNN con k=3 -> Número de clases
knn = KNeighborsClassifier(n_neighbors=3)

# Entreno el modelo
knn.fit(X_train, y_train)

# Hago predicciones
y_pred = knn.predict(X_test)

### 5.3 Evaluación del desempeño

In [ ]:
# Evalúo el rendimiento del modelo
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Interpretación:
  
- **Exactitud (Accuracy)**: El modelo acertó en el 79.6% de los casos.
- **Precisión (Precision)**: De todas las predicciones positivas, el 75% fueron correctas en clase, el 74% en clase 2 y 100% clase 3.
- **Sensibilidad (Recall)**: El modelo identificó correctamente el 79% de los casos positivos en clase 1, 81% en clase 2 y 79% en clase 3.
- **F1-Score**: Una medida de balance entre precisión y sensibilidad, alcanzando 0.77 en clase 1, 0.77 en clase 2 y 0.88 en clase 3.

El modelo de KNN con k=3 tuvo una precisión del 79.6%, acertando en la mayoría de los casos.
- Clase 3 tuvo 100% de precisión, pero su recall fue 79%, lo que indica que identificó la mayoría de los casos correctamente.
- Clases 1 y 2 fueron más equilibradas, con precisión y recall entre 75% y 81%.
En general, el modelo funcionó bien, aunque podría mejorar la identificación de algunas muestras en clases 1 y 2.

## **6.  Tuneo de hiperparámetros**

En esta sección debe modificar probar cómo la modificación de un hiperparámetro del modelo elegido en punto 5) afecta en los resultados.

Justifique y realice el experimento en esta sección.

In [ ]:
# Probar diferentes valores de k
for k in [1, 5, 7, 10]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)

    print(f"\nResultados para k={k}:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
    print(classification_report(y_test, y_pred))

**Análisis de la modificación del hiperparámetro k**

Al cambiar k, observamos cómo afecta la precisión y el rendimiento de la clasificación:
- **Con k=1:**
  - La precisión es 79.6%, ligeramente más baja que con valores mayores de k.
- **Con k=5:**
  - La precisión mejora a 83.3%, mostrando un mejor equilibrio entre las clases.
  - La clase 3 mantiene alta precisión (1.00), pero con menor sensibilidad (0.79).
- **Con k=7 y k=10:**
  - La precisión se mantiene en 83.3%, lo que indica estabilidad.
  - La clase 3 mejora su f1-score con k=10 (0.89), lo que sugiere que elegir un k ligeramente mayor ayuda a la clasificación de esta clase.



AMPLIANDO LA INTERPRETACION k=5:
  
- **Exactitud (Accuracy)**: El modelo acertó en el 83% de los casos.
- **Precisión (Precision)**: De todas las predicciones positivas, el 80% fueron correctas en clase, el 78% en clase 2 y 100% clase 3.
- **Sensibilidad (Recall)**: El modelo identificó correctamente el 84% de los casos positivos en clase 1, 86% en clase 2 y 79% en clase 3.
- **F1-Score**: Una medida de balance entre precisión y sensibilidad, alcanzando 0.82 en clase 1, 0.82 en clase 2 y 0.88 en clase 3.

### **CONCLUSÍON DE KNN PARA DISTINTOS VALORES DE K**
Con KNN k=5, el modelo mejora su precisión y estabilidad, alcanzando una exactitud del 83% frente al 79.6% de k=3.
- Clase 1 y 2 muestran mejor precisión y recall con k=5, lo que indica que el modelo identifica mejor estas categorías.
- Clase 3 mantiene 100% de precisión, pero su recall sigue en 79%, lo que sugiere que no todos los casos fueron correctamente reconocidos.

Por lo tanto, si bien k=3 parecía una opción lógica al inicio, los resultados muestran que k=5 o k=7 son mejores elecciones, ya que ofrecen mayor estabilidad y precisión al clasificar los tres tipos de vino. Además, es posible que la división entre clases haya sido afectada por la eliminación de algunas features o la presencia de outliers, lo que podría haber influido en el rendimiento del modelo.


## **7. Conclusiones**

Fundamente, justifique con sus palabras.

Después de probar **Logistic Regression** y **KNN con k=5**, noté que ambos modelos funcionan bien, pero tienen diferencias minimas en cómo clasifican los vinos.

* Muy importante tener en cuenta que LR clasifica solo 2 clases ya que es binaria a diferencia de KNN que se tienen en cuenta todas las clases.
* Ambas tienen una exactitud (acurracy), predicción correcta, del 83%.
* En el caso de la precisión LG es mejor en la clase 1 (0.83 > 0.8).
* La sensibilidad (recall) en en KNN es levemente superior.
* Y por ultimo, F1-Score es similar en ambos. Por lo tanto, al ser relativamente alto indica un buen balance entre precisión y recall en LR y KNN.

Me he dado cuenta que KNN es mas sensible a valores atipicos (que los he dejado intactos) ya que clasifica los datos basandose en los vecinos mas cercanos. Pienso esto porque K=3 (un poco pequeño), que en la logica seria el numero de clusters ideal ya que es la cantidad de clases de vinos en el dataset, tiene una exactitud menor que k=5.

Si revisara nuevamente los outliers y los trabajara de manera correcta, KNN seria el clasificador mas adecuando en mi caso ya que tiene 3 clases bien marcadas.

Esta comparación me ayudó a entender que la elección de un modelo depende de lo que quiero priorizar en la clasificación.


## Referencias

* Clases de la materia
* Chat GPT
* Copilot (IA de mi computadora)
* UCI plataforma del Dataset

# 🍷 **Clasificación de vinos con Redes Neuronales** - TP1 AA2


## **Introducción**

En este trabajo práctico se abordará un problema de aprendizaje supervisado utilizando redes neuronales en PyTorch.

El objetivo es comparar el desempeño de una red neuronal con modelos de aprendizaje automático previamente utilizados en la materia AA1.

Para ello, se trabajará con el dataset **Wine**, que consiste en la clasificación de distintos tipos de vino a partir de sus propiedades químicas.

## Dataset: Wine

El dataset Wine contiene información sobre diferentes muestras de vino, caracterizadas por variables químicas como alcohol, acidez, magnesio, entre otras.

### Tipo de problema
Clasificación multiclase.

### Objetivo
Predecir a qué clase de vino pertenece cada muestra.

### Características
- Variables numéricas
- 3 clases posibles

## Objetivos

- Implementar una red neuronal en PyTorch para resolver el problema de clasificación.
- Entrenar el modelo y evaluar su desempeño.
- Analizar métricas en entrenamiento y validación.
- Realizar experimentos modificando la arquitectura.
- Comparar los resultados con modelos tradicionales utilizados en AA1.

In [ ]:
# Fijar semilla para reproducibilidad (evita que cambien loss y accuracy en cada ejecución) -> codigo realizado con Gemini
import torch
import numpy as np
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Librerías principales
import pandas as pd

# Dataset y preprocesamiento
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# PyTorch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader


# numpy y pandas → manejo de datos
# load_wine → dataset Wine
# train_test_split → dividir en entrenamiento y prueba
# StandardScaler → normalizar variables
# torch / nn → red neuronal en PyTorch
# TensorDataset / DataLoader → preparar datos para entrenamiento por batches

# SEED → fija la aleatoriedad para que los resultados sean siempre los mismos

In [ ]:
# Cargo el dataset Wine
wine = load_wine()

# Separo variables de entrada (X) y variable objetivo (y)
X = wine.data
y = wine.target

# Muestro dimensiones
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

# X → variables predictoras
# y → clase de vino
# shape muestra:
# - cantidad de muestras
# - cantidad de variables

Forma de X: (178, 13)
Forma de y: (178,)


In [ ]:
# Convierto a DataFrame para explorar de una forma mas comoda
df = pd.DataFrame(X, columns=wine.feature_names)
df["target"] = y

# Muestro las primeras filas
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


## Exploración inicial del dataset

En esta sección cargo el dataset Wine y realizo una exploración básica.

Trabajo con:

- **X**: variables de entrada
- **y**: clase objetivo

El dataset presenta un problema de **clasificación multiclase**, ya que existen tres categorías posibles de vino.

In [ ]:
# Cuento cuántos ejemplos hay por clase
df["target"].value_counts().sort_index()

,count
target,
0,59
1,71
2,48


In [ ]:
# @title
import plotly.express as px

fig_count = px.histogram(df, x=df["target"].astype(str), title="Distribución de clases en el dataset Wine",
                        color_discrete_sequence=['#8C2F39', '#D4A88E'],
                        template="plotly_white")

fig_count.update_layout(
    title_x=0.5,
    xaxis_title="Clase",
    yaxis_title="Cantidad de muestras",
    font=dict(size=14),
    bargap=0.2 # Separar las barras
)

fig_count.show()

# Este gráfico permite observar si las clases están balanceadas o no

### Análisis de la distribución de clases

Se observa que las tres clases están relativamente balanceadas, aunque la clase 1 presenta una mayor cantidad de muestras en comparación con las demás.

Sin embargo, la diferencia no es extrema, por lo que no se espera un impacto significativo en el entrenamiento del modelo.

En general, este dataset puede considerarse suficientemente balanceado para aplicar modelos de clasificación sin necesidad de técnicas adicionales de balanceo.

## División de datos

Se divide el dataset en dos conjuntos:

- Entrenamiento: utilizado para ajustar los parámetros del modelo.
- Test: utilizado para evaluar el desempeño en datos no vistos.

Se utiliza el parámetro `stratify` para mantener la proporción de clases en ambos conjuntos.

In [ ]:
# Divido el dataset en entrenamiento y test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para test
    random_state=42,
    stratify=y           # mantiene proporción de clases
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (142, 13)
Test: (36, 13)


## Normalización de datos

Se aplica StandardScaler para estandarizar las variables.

Esto permite que todas las características tengan media 0 y desviación estándar 1.

La normalización es importante en redes neuronales porque:
- mejora la estabilidad del entrenamiento
- acelera la convergencia

El escalador se ajusta únicamente con los datos de entrenamiento para evitar fuga de información.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Creo el escalador
scaler = StandardScaler()

# Ajusto SOLO con train
X_train = scaler.fit_transform(X_train)

# Aplico al test
X_test = scaler.transform(X_test)

## Conversión a tensores

Los datos se convierten a tensores de PyTorch para poder ser utilizados por la red neuronal.

- Las variables de entrada se convierten a float32.
- Las etiquetas se convierten a long, ya que son necesarias para la función de pérdida de clasificación.

In [ ]:
# Convierto a tensores
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

## DataLoader

Se utilizan DataLoader para trabajar con los datos en batches.

Esto permite:
- entrenar el modelo de forma más eficiente
- mejorar la generalización
- manejar datasets más grandes

Se utiliza shuffle en el conjunto de entrenamiento para evitar que el modelo aprenda patrones del orden de los datos.

In [ ]:
# Creo datasets
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

# Creo DataLoaders
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=16)

## Red neuronal utilizada - Modelo 1

Para resolver este problema armé una red neuronal simple usando PyTorch.

Como el dataset Wine tiene variables numéricas y no imágenes, en este caso no hace falta usar una red convolucional. Por eso elegí una red totalmente conectada (fully connected), que es más adecuada para este tipo de datos.

La idea es que la red reciba las variables de entrada, las procese en una capa oculta y finalmente devuelva una predicción sobre la clase del vino.

Elegí una arquitectura simple para poder entender bien su funcionamiento y después comparar cómo cambia el rendimiento al modificar la cantidad de neuronas o de capas.

In [ ]:
class WineNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Primera capa: recibe las 13 variables del dataset
        self.fc1 = nn.Linear(13, 16)

        # Segunda capa: salida final con 3 clases posibles
        self.fc2 = nn.Linear(16, 3)

    def forward(self, x):
        # Paso 1: primera transformación lineal
        x = self.fc1(x)

        # Paso 2: función de activación
        x = torch.relu(x)

        # Paso 3: capa de salida
        x = self.fc2(x)

        return x

### Explicación de la arquitectura

En este modelo usé:

- una capa de entrada de 13 variables, porque el dataset Wine tiene 13 características
- una capa oculta de 16 neuronas
- una capa de salida de 3 neuronas, porque hay 3 clases de vino

Elegí usar 16 neuronas como un valor intermedio para empezar. No hay una regla exacta para esto, pero se suelen probar números como 8, 16 o 32. En este caso me pareció una buena opción porque no es ni muy chico (lo que podría generar subajuste) ni muy grande (lo que podría llevar a sobreajuste). Más adelante voy a comparar con otras configuraciones.

También usé la función de activación ReLU en la capa oculta, porque ayuda a que la red pueda aprender relaciones no lineales entre los datos.

En la capa de salida no agregué ninguna activación extra, porque después voy a usar `CrossEntropyLoss`, que ya trabaja directamente con la salida del modelo.

In [ ]:
model = WineNet()
print(model)

WineNet(
  (fc1): Linear(in_features=13, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=3, bias=True)
)


### Justificación

Elegí empezar con una red simple porque primero quiero tener una base clara y fácil de interpretar.

Me parece una buena primera opción para este dataset, ya que no tiene demasiadas variables y el problema no parece requerir una arquitectura demasiado compleja desde el inicio.

Más adelante voy a probar otras configuraciones para comparar resultados y observar si aparecen señales de subajuste o sobreajuste.

## Función de pérdida y optimizador

Para entrenar la red usé `CrossEntropyLoss`, porque este es un problema de clasificación multiclase.

También usé el optimizador `Adam`, que suele funcionar bien y facilita bastante el entrenamiento.

El learning rate lo fijé en 0.001 para que el aprendizaje sea más estable.

In [ ]:
# Función de pérdida para clasificación multiclase
loss_fn = nn.CrossEntropyLoss()

# Optimizador
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Métrica de evaluación

Además de la pérdida, voy a usar accuracy para medir el rendimiento del modelo.

Esta métrica me muestra qué proporción de ejemplos fue clasificada correctamente.

In [ ]:
def accuracy_fn(y_pred, y_true):
    preds = torch.argmax(y_pred, dim=1)
    correct = (preds == y_true).float().mean()
    return correct

## Entrenamiento y prueba del modelo

En esta etapa entreno la red neuronal usando los datos de entrenamiento.

La idea es que en cada época el modelo haga predicciones, calcule el error y ajuste sus pesos para mejorar.

También voy guardando la loss y la accuracy para poder analizar cómo evoluciona el modelo.

---
Después del entrenamiento, evalué el modelo con el conjunto de test.

Esto me permite ver cómo se comporta con datos no vistos y no solo con los datos de entrenamiento.

Voy a calcular la loss y la accuracy en test para comparar ambos resultados.


In [ ]:
epochs = 20

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

for epoch in range(epochs):

    # Modo entrenamiento
    model.train()

    # Forward (predicciones)
    y_pred = model(X_train)

    # Calcular loss
    loss = loss_fn(y_pred, y_train)

    # Calcular accuracy
    acc = accuracy_fn(y_pred, y_train)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Guardar métricas de entrenamiento
    train_losses.append(loss.item())
    train_accuracies.append(acc.item())

    # Evaluación en test al final de cada época
    model.eval() # Poner el modelo en modo evaluación
    with torch.no_grad():
        y_test_pred = model(X_test)
        current_test_loss = loss_fn(y_test_pred, y_test)
        current_test_acc = accuracy_fn(y_test_pred, y_test)
    test_losses.append(current_test_loss.item()) # Guardar loss de test
    test_accuracies.append(current_test_acc.item()) # Guardar accuracy de test

    # Mostrar resultados
    print(f"Epoch {epoch}: train_loss = {loss:.4f}, train_accuracy = {acc:.4f}, test_loss = {current_test_loss:.4f}, test_accuracy = {current_test_acc:.4f}")

Epoch 0: train_loss = 1.1833, train_accuracy = 0.0845, test_loss = 1.1754, test_accuracy = 0.1389
Epoch 1: train_loss = 1.1760, train_accuracy = 0.0915, test_loss = 1.1678, test_accuracy = 0.1667
Epoch 2: train_loss = 1.1688, train_accuracy = 0.1056, test_loss = 1.1602, test_accuracy = 0.1667
Epoch 3: train_loss = 1.1616, train_accuracy = 0.1197, test_loss = 1.1527, test_accuracy = 0.1944
Epoch 4: train_loss = 1.1545, train_accuracy = 0.1268, test_loss = 1.1452, test_accuracy = 0.2222
Epoch 5: train_loss = 1.1475, train_accuracy = 0.1549, test_loss = 1.1378, test_accuracy = 0.2222
Epoch 6: train_loss = 1.1405, train_accuracy = 0.1972, test_loss = 1.1305, test_accuracy = 0.2222
Epoch 7: train_loss = 1.1337, train_accuracy = 0.2183, test_loss = 1.1232, test_accuracy = 0.2500
Epoch 8: train_loss = 1.1269, train_accuracy = 0.2254, test_loss = 1.1160, test_accuracy = 0.2500
Epoch 9: train_loss = 1.1202, train_accuracy = 0.2394, test_loss = 1.1088, test_accuracy = 0.2500
Epoch 10: train_loss

### Análisis del entrenamiento

Durante el entrenamiento observé que  loss disminuye de forma progresiva en cada época, lo que indica que el modelo está aprendiendo.

La accuracy también mejora de manera gradual, pasando de valores muy bajos (~0.08) hasta aproximadamente 0.48 en entrenamiento. Esto muestra que el modelo está empezando a captar algunos patrones del dataset, aunque todavía no logra un rendimiento alto.

En general, el modelo aprende, pero lo hace de forma lenta y limitada.

---

### Evaluación en test

Para evaluar el rendimiento real del modelo, utilicé el conjunto de test.

Puse el modelo en modo evaluación con `model.eval()` y utilicé `torch.no_grad()` para evitar el cálculo de gradientes, ya que en esta etapa no se ajustan los pesos, solo se analiza cómo predice.

Esto me permite comparar el comportamiento del modelo con datos que no fueron utilizados durante el entrenamiento.

---

### Comparación entre entrenamiento y test

Al comparar los resultados, observo que:

- la accuracy en entrenamiento llega aproximadamente a 0.48
- la accuracy en test alcanza aproximadamente 0.50

Además, la loss en ambos casos sigue siendo relativamente alta.

Esto indica que el modelo está aprendiendo, pero todavía no logra representar bien los patrones del dataset.

No se observa sobreajuste, ya que el rendimiento en entrenamiento y test es similar.

En este caso, se trata de un modelo con subajuste (underfitting), probablemente porque la arquitectura es demasiado simple o no se entrenó lo suficiente.

---

### Posibles mejoras

Para mejorar el rendimiento del modelo, podría:

- aumentar la cantidad de épocas
- utilizar una arquitectura más compleja (más neuronas o más capas)
- ajustar hiperparámetros como el learning rate

## Modelo 2

Como en el primer modelo observé un rendimiento bajo tanto en entrenamiento como en test, decidí probar una arquitectura un poco más completa.

En este segundo modelo agregué:

- una primera capa oculta de 32 neuronas
- una segunda capa oculta de 16 neuronas

La idea es darle a la red mayor capacidad para aprender relaciones entre las variables del dataset y después comparar si esto mejora el rendimiento.

Con este cambio busco observar si el modelo anterior estaba quedando corto y si una arquitectura un poco más profunda permite reducir el subajuste.

In [ ]:
class WineNet2(nn.Module):
    def __init__(self):
        super().__init__()

        # Primera capa oculta
        self.fc1 = nn.Linear(13, 32)

        # Segunda capa oculta
        self.fc2 = nn.Linear(32, 16)

        # Capa de salida
        self.fc3 = nn.Linear(16, 3)

    def forward(self, x):
        # Primera transformación + activación
        x = self.fc1(x)
        x = torch.relu(x)

        # Segunda transformación + activación
        x = self.fc2(x)
        x = torch.relu(x)

        # Salida final
        x = self.fc3(x)

        return x

### Explicación de la arquitectura

En este segundo modelo usé:

- una capa de entrada de 13 variables
- una primera capa oculta de 32 neuronas
- una segunda capa oculta de 16 neuronas
- una capa de salida de 3 neuronas

La diferencia con el modelo anterior es que ahora la red tiene una capa más y más neuronas al principio.

Elegí esta configuración para probar una arquitectura más potente, pero todavía bastante simple y razonable para este dataset.

In [ ]:
model2 = WineNet2()

loss_fn2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

print(model2)

WineNet2(
  (fc1): Linear(in_features=13, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=3, bias=True)
)


### Justificación

En este modelo mantuve ReLU como función de activación y CrossEntropyLoss como función de pérdida, porque siguen siendo adecuadas para este problema de clasificación multiclase.

El principal cambio fue aumentar la complejidad de la red para ver si mejora el aprendizaje y si logra capturar mejor los patrones del dataset.

## Entrenamiento y prueba del modelo 2

Ahora entreno el segundo modelo para comparar su comportamiento con el modelo anterior.

Voy a guardar nuevamente la loss y la accuracy para analizar si esta arquitectura mejora el rendimiento tanto en entrenamiento como en test.

---
Después del entrenamiento evalué el segundo modelo con el conjunto de test para ver si la mejora observada en entrenamiento también se mantiene en datos no vistos.

Esto me permite comparar si el modelo realmente generaliza bien o si solo aprendió demasiado los datos de entrenamiento.


In [ ]:
epochs = 50

train_losses_2 = []
train_accuracies_2 = []
test_losses_2 = []
test_accuracies_2 = []

for epoch in range(epochs):

    model2.train()

    y_pred_2 = model2(X_train)

    loss_2 = loss_fn2(y_pred_2, y_train)
    acc_2 = accuracy_fn(y_pred_2, y_train)

    optimizer2.zero_grad()
    loss_2.backward()
    optimizer2.step()

    train_losses_2.append(loss_2.item())
    train_accuracies_2.append(acc_2.item())

    # Evaluación en test al final de cada época
    model2.eval() # Poner el modelo en modo evaluación
    with torch.no_grad():
        y_test_pred_2 = model2(X_test)
        current_test_loss_2 = loss_fn2(y_test_pred_2, y_test)
        current_test_acc_2 = accuracy_fn(y_test_pred_2, y_test)
    test_losses_2.append(current_test_loss_2.item()) # Guardar loss de test
    test_accuracies_2.append(current_test_acc_2.item()) # Guardar accuracy de test

    print(f"Epoch {epoch}: train_loss = {loss_2:.4f}, train_accuracy = {acc_2:.4f}, test_loss = {current_test_loss_2:.4f}, test_accuracy = {current_test_acc_2:.4f}")

Epoch 0: train_loss = 1.0925, train_accuracy = 0.4296, test_loss = 1.0743, test_accuracy = 0.4444
Epoch 1: train_loss = 1.0820, train_accuracy = 0.4648, test_loss = 1.0628, test_accuracy = 0.4444
Epoch 2: train_loss = 1.0717, train_accuracy = 0.4789, test_loss = 1.0516, test_accuracy = 0.5000
Epoch 3: train_loss = 1.0616, train_accuracy = 0.5000, test_loss = 1.0406, test_accuracy = 0.5000
Epoch 4: train_loss = 1.0515, train_accuracy = 0.5423, test_loss = 1.0298, test_accuracy = 0.5556
Epoch 5: train_loss = 1.0416, train_accuracy = 0.5634, test_loss = 1.0190, test_accuracy = 0.5556
Epoch 6: train_loss = 1.0318, train_accuracy = 0.5704, test_loss = 1.0081, test_accuracy = 0.5556
Epoch 7: train_loss = 1.0221, train_accuracy = 0.5704, test_loss = 0.9974, test_accuracy = 0.5833
Epoch 8: train_loss = 1.0126, train_accuracy = 0.5775, test_loss = 0.9867, test_accuracy = 0.6111
Epoch 9: train_loss = 1.0032, train_accuracy = 0.5775, test_loss = 0.9762, test_accuracy = 0.6111
Epoch 10: train_loss

En el modelo 1 usé 20 épocas y vi que la red casi no mejoraba.

En este segundo modelo decidí aumentar la cantidad de épocas para darle más oportunidad de aprender, ya que ahora la arquitectura es más compleja.

### Análisis del entrenamiento del modelo 2

En este segundo modelo se observa una mejora clara respecto al modelo anterior.

La loss disminuye de forma sostenida a lo largo de las épocas, mientras que la accuracy en entrenamiento aumenta progresivamente, pasando de aproximadamente 0.43 hasta cerca de 0.89.

Esto indica que el modelo logra aprender los patrones del dataset de forma mucho más efectiva que el modelo anterior.

El aprendizaje es gradual pero constante, lo que sugiere que la arquitectura tiene mayor capacidad y está mejor ajustada al problema.

---
### Comparación entre modelos

Al comparar ambos modelos, se observa una diferencia clara en el rendimiento.

Modelo 1:
- accuracy en entrenamiento ≈ 0.48
- accuracy en test ≈ 0.50  

Presentaba subajuste, ya que no lograba aprender correctamente los patrones del dataset.

Modelo 2:
- accuracy en entrenamiento ≈ 0.89
- accuracy en test ≈ 0.88  

Muestra un rendimiento significativamente mejor tanto en entrenamiento como en test.

Esto indica que aumentar la cantidad de capas y neuronas permitió que el modelo tenga mayor capacidad de aprendizaje.

Además, la diferencia entre entrenamiento y test es pequeña, lo que sugiere que el modelo generaliza bien y no presenta un sobreajuste significativo.

En conclusión, el modelo 2 resulta mucho más adecuado para este problema.


## Cierre visual del trabajo

Para cerrar el análisis, armé algunos gráficos interactivos para comparar el comportamiento de los dos modelos.

Solicité a Gemini la generación de gráficos de las curvas de Accuracy y Loss para el entrenamiento y test de ambos modelos, con el fin de analizar el subajuste y sobreajuste.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

### Curvas de Accuracy (entrenamiento vs. test)

In [ ]:
# @title
# Armo un DataFrame unificado
df_acc = pd.DataFrame({
    "epoch": list(range(1, len(train_accuracies) + 1)) * 2 +
             list(range(1, len(train_accuracies_2) + 1)) * 2,

    "accuracy": train_accuracies + test_accuracies +
                train_accuracies_2 + test_accuracies_2,

    "tipo": (["Train"] * len(train_accuracies) +
             ["Test"] * len(test_accuracies) +
             ["Train"] * len(train_accuracies_2) +
             ["Test"] * len(test_accuracies_2)),

    "modelo": (["Modelo 1"] * len(train_accuracies) +
               ["Modelo 1"] * len(test_accuracies) +
               ["Modelo 2"] * len(train_accuracies_2) +
               ["Modelo 2"] * len(test_accuracies_2))
})

# Creo el gráfico
fig = px.line(
    df_acc,
    x="epoch",
    y="accuracy",
    color="modelo",
    line_dash="tipo",
    markers=True,
    title="Evolución de Accuracy (Train vs Test)",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="Época",
    yaxis_title="Accuracy",
    title_x=0.5
)

fig.show()

#### Análisis de las curvas de accuracy

En este gráfico comparo la evolución de la accuracy en entrenamiento y test para ambos modelos.

En el modelo 1 (líneas azules), la accuracy aumenta de forma lenta y se mantiene en valores bajos, llegando aproximadamente a 0.48 en entrenamiento y 0.50 en test. Esto confirma que el modelo no logra aprender bien los patrones del dataset.

En el modelo 2 (líneas rojas), la accuracy mejora de forma sostenida a lo largo de las épocas, alcanzando valores cercanos a 0.89 tanto en entrenamiento como en test. Esto indica que el modelo tiene un aprendizaje mucho más efectivo.

Además, en el modelo 2 las curvas de entrenamiento y test se mantienen bastante cercanas, lo que sugiere que el modelo generaliza bien y no presenta sobreajuste.

En conclusión, el modelo 1 presenta un bajo rendimiento, mientras que el modelo 2 logra un buen nivel de precisión y un aprendizaje más adecuado para el problema.

### Curvas de Loss (entrenamiento vs. test)

In [ ]:
# @title
df_loss = pd.DataFrame({
    "epoch": list(range(1, len(train_losses) + 1)) * 2 +
             list(range(1, len(train_losses_2) + 1)) * 2,

    "loss": train_losses + test_losses +
            train_losses_2 + test_losses_2,

    "tipo": (["Train"] * len(train_losses) +
             ["Test"] * len(test_losses) +
             ["Train"] * len(train_losses_2) +
             ["Test"] * len(test_losses_2)),

    "modelo": (["Modelo 1"] * len(train_losses) +
               ["Modelo 1"] * len(test_losses) +
               ["Modelo 2"] * len(train_losses_2) +
               ["Modelo 2"] * len(test_losses_2))
})

fig = px.line(
    df_loss,
    x="epoch",
    y="loss",
    color="modelo",
    line_dash="tipo",
    markers=True,
    title="Evolución de Loss (Train vs Test)",
    template="plotly_white"
)

fig.update_layout(
    xaxis_title="Época",
    yaxis_title="Loss",
    title_x=0.5
)

fig.show()

#### Análisis de las curvas de loss

En este gráfico comparo la evolución de la loss en entrenamiento y test para ambos modelos.

En el modelo 1 (líneas azules), observo que la loss disminuye muy poco a lo largo de las épocas y se mantiene en valores relativamente altos. Además, las curvas de entrenamiento y test son muy parecidas. Esto indica que el modelo no logra aprender bien los patrones del dataset, es decir, presenta subajuste.

En el modelo 2 (líneas rojas), la loss disminuye de forma mucho más marcada a lo largo de las épocas, tanto en entrenamiento como en test. Esto muestra que el modelo aprende de manera más efectiva.

También observo que las curvas de entrenamiento y test en el modelo 2 se mantienen cercanas entre sí, lo que indica que no hay sobreajuste. El modelo logra generalizar bien a datos no vistos.

En conclusión, el modelo 1 presenta subajuste, mientras que el modelo 2 logra un mejor aprendizaje sin evidencias de sobreajuste, siendo más adecuado para este problema.

## Conclusión

En este trabajo pude observar cómo influye la arquitectura de una red neuronal en su rendimiento.

En el modelo 1 utilicé una estructura simple que no logró aprender correctamente los patrones del dataset, lo que se reflejó en valores bajos de accuracy y una loss relativamente alta. A partir del análisis de las curvas, pude identificar que se trataba de un caso de subajuste.

Luego, al aumentar la complejidad en el modelo 2, el rendimiento mejoró notablemente. La loss disminuyó de forma sostenida y la accuracy alcanzó valores altos tanto en entrenamiento como en test, lo que indica un aprendizaje más efectivo.

Además, al comparar las curvas de entrenamiento y test, observé que se mantienen cercanas, lo que sugiere que el modelo generaliza bien y no presenta sobreajuste.

Este trabajo me permitió entender en la práctica la importancia de la arquitectura y los hiperparámetros en el entrenamiento de redes neuronales, así como la utilidad de analizar métricas como la loss y la accuracy para evaluar el comportamiento del modelo.

## Aplicación en un contexto real

Si este modelo se aplicara en un entorno real, podría utilizarse para automatizar la clasificación de vinos a partir de sus características químicas.

Por ejemplo, en una bodega o laboratorio, este tipo de modelo permitiría identificar rápidamente el tipo de vino sin depender exclusivamente de un análisis manual, lo que ayudaría a reducir tiempos y hacer el proceso más consistente.

En base a los resultados obtenidos, el modelo 2 alcanza una accuracy alta (~0.91 en test), lo que indica que tiene un buen nivel de precisión y que podría ser útil como herramienta de apoyo en la toma de decisiones.

Además, como el rendimiento en entrenamiento y test es similar, el modelo muestra una buena capacidad de generalización, lo que es importante si se quiere aplicar con datos nuevos.

Sin embargo, también es importante tener en cuenta que el modelo no es perfecto. Antes de usarlo en producción, sería necesario validarlo con más datos y analizar qué tan críticos son los posibles errores de clasificación.

En este sentido, lo más adecuado sería usarlo como un sistema de apoyo que complemente el trabajo humano, en lugar de reemplazarlo completamente.

## **Comparación con modelos de AA1**

En el trabajo anterior (AA1) utilicé modelos de aprendizaje automático como Logistic Regression y K-Nearest Neighbors (KNN), evaluados mediante accuracy.

Ambos modelos lograron un rendimiento similar, con una accuracy aproximada del 83%, mostrando un buen desempeño general en la clasificación del dataset.

Al compararlos con la red neuronal implementada en este trabajo (AA2), se observa que el modelo 2 alcanza una accuracy mayor (alrededor de 0.88 - 0.91), lo que indica una mejora en el rendimiento.

Sin embargo, también noté que los modelos de AA1 son más simples de implementar y requieren menos ajuste, mientras que la red neuronal necesita definir arquitectura, hiperparámetros y proceso de entrenamiento.

Por otro lado, la red neuronal tiene mayor capacidad para aprender patrones complejos, lo que se refleja en su mejor desempeño en este caso.

En conclusión, los modelos de AA1 resultan adecuados para problemas simples o cuando se busca una solución rápida, mientras que las redes neuronales permiten obtener mejores resultados cuando se ajustan correctamente, aunque con mayor complejidad.

Esta comparación me permitió entender que no siempre el modelo más complejo es necesario, pero sí puede marcar la diferencia cuando el problema lo requiere.

## Uso de herramientas de IA y recursos

Durante el desarrollo de este trabajo utilicé ChatGPT como herramienta de apoyo para aclarar conceptos, entender mejor el flujo de entrenamiento y resolver dudas puntuales. Lo utilicé como un tutor académico para la materia AA2, no para que resuelva el trabajo por mí.

También utilicé la asistencia de Gemini en Google Colab para mejorar la parte visual, principalmente en la elección de colores para los gráficos.

Además, me basé en los materiales de la cursada, como los videos vistos en clase, el notebook de PyTorch en inglés, los tutoriales de Microsoft Learn, el tutorial oficial de PyTorch y algunos artículos teóricos.

También tuve en cuenta la bibliografía recomendada, como *Deep Learning with PyTorch* y *Deep Learning for Coders with fastai and PyTorch*.

No adjunto el enlace completo del chat porque utilicé un solo chat para toda la materia. De todas formas, todas las decisiones, implementaciones y análisis fueron realizados por mí en base a lo aprendido.